# Projet Data Science : Evolution des statistiques des Pokemon par generation

## Problematique
Les generations recentes introduisent-elles des Pokemon aux statistiques plus elevees ou plus specialises ?

## Hypotheses
- H1 : Le total des statistiques de base (BST) augmente avec les generations.
- H2 : Les profils statistiques deviennent plus specialises.

Les Pokemon sont etudies selon leur generation d'introduction. Les formes alternatives ne sont pas comptees comme de nouvelles especes. Les legendaires et fabuleux seront egalement analyses separement.

## 1. Importation des bibliotheques

Nous utilisons Pandas pour manipuler les donnees, Matplotlib et Seaborn pour les graphiques, SciPy pour les tests statistiques et Requests pour interroger l'API.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import json
from scipy import stats
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

BASE_URL = 'https://pokeapi.co/api/v2/'
CACHE_DIR = 'pokeapi_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print('Bibliotheques chargees avec succes !')

## 2. Recuperation des donnees depuis PokéAPI

On recupere les especes introduites dans chacune des 9 generations. Pour chaque espece, on selectionne sa forme par defaut et ses six statistiques de base.

Un cache local permet d'eviter de telecharger plusieurs fois les memes informations.

In [ ]:
session = requests.Session()


def get_api(url):
    filename = os.path.join(CACHE_DIR, str(abs(hash(url))) + '.json')

    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            return json.load(f)

    response = session.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f)

    time.sleep(0.05)
    return data


pokemon_data = []

for generation in range(1, 10):
    print(f'Recuperation generation {generation}...')

    gen_data = get_api(f'{BASE_URL}generation/{generation}/')

    for species in gen_data['pokemon_species']:
        species_data = get_api(species['url'])

        default_variety = next(
            (v for v in species_data['varieties'] if v['is_default']),
            None
        )

        if default_variety is None:
            continue

        pokemon = get_api(default_variety['pokemon']['url'])

        row = {
            'id': species_data['id'],
            'name': species_data['name'],
            'generation': generation,
            'is_legendary': species_data['is_legendary'],
            'is_mythical': species_data['is_mythical'],
            'types': ', '.join(t['type']['name'] for t in pokemon['types'])
        }

        for stat in pokemon['stats']:
            row[stat['stat']['name']] = stat['base_stat']

        pokemon_data.append(row)


df = pd.DataFrame(pokemon_data)
df = df.drop_duplicates(subset='id').reset_index(drop=True)

print('\nCollecte terminee !')
print('Nombre de Pokemon :', len(df))
display(df.head())

## 3. Nettoyage et preparation des donnees

Le BST (Base Stat Total) correspond a la somme des six statistiques : PV, Attaque, Defense, Attaque Speciale, Defense Speciale et Vitesse.

Pour mesurer la specialisation, on utilise le coefficient de variation des six statistiques : ecart-type divise par moyenne. Plus il est eleve, plus le profil est contraste.

Cette mesure ne represente pas directement la performance en combat, mais permet de comparer les profils.

In [ ]:
stat_cols = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']

print('Valeurs manquantes :')
print(df[stat_cols].isna().sum())

# Suppression des lignes incompletes
df = df.dropna(subset=stat_cols).copy()

# Total des statistiques de base
df['BST'] = df[stat_cols].sum(axis=1)

# Specialisation : coefficient de variation
df['specialization'] = (
    df[stat_cols].std(axis=1, ddof=0) /
    df[stat_cols].mean(axis=1)
)

# Ecart entre la statistique maximale et minimale
df['stat_gap'] = df[stat_cols].max(axis=1) - df[stat_cols].min(axis=1)

# Categorie pour analyses complementaires
df['legendary_status'] = np.where(
    df['is_legendary'], 'Legendary',
    np.where(df['is_mythical'], 'Mythical', 'Normal')
)

print('Dimensions du dataset :', df.shape)
display(df.head())
print(df.describe())

## 4. Analyse exploratoire

Nous comparons le nombre de Pokemon par generation, le BST moyen et la specialisation moyenne.

In [ ]:
summary = df.groupby('generation').agg(
    pokemon_count=('id', 'count'),
    BST_mean=('BST', 'mean'),
    BST_median=('BST', 'median'),
    BST_std=('BST', 'std'),
    specialization_mean=('specialization', 'mean'),
    specialization_median=('specialization', 'median')
).round(2)

display(summary)

### 4.1 Nombre de Pokemon par generation

In [ ]:
sns.countplot(data=df, x='generation', color='steelblue')
plt.title('Nombre de Pokemon par generation')
plt.xlabel('Generation')
plt.ylabel('Nombre de Pokemon')
plt.show()

### 4.2 Evolution du total des statistiques (BST)

On observe la moyenne et la distribution des BST pour chaque generation.

In [ ]:
sns.boxplot(data=df, x='generation', y='BST', color='skyblue')
plt.title('Distribution du BST par generation')
plt.xlabel('Generation')
plt.ylabel('Total des statistiques de base')
plt.show()

summary['BST_mean'].plot(marker='o', color='darkblue')
plt.title('Evolution du BST moyen par generation')
plt.xlabel('Generation')
plt.ylabel('BST moyen')
plt.xticks(range(1, 10))
plt.show()

### 4.3 Evolution de la specialisation

Un coefficient de variation eleve indique une repartition plus inegale des statistiques d'un Pokemon.

In [ ]:
sns.boxplot(data=df, x='generation', y='specialization', color='orange')
plt.title('Specialisation des Pokemon par generation')
plt.xlabel('Generation')
plt.ylabel('Coefficient de variation')
plt.show()

summary['specialization_mean'].plot(marker='o', color='darkred')
plt.title('Specialisation moyenne par generation')
plt.xlabel('Generation')
plt.ylabel('Coefficient de variation moyen')
plt.xticks(range(1, 10))
plt.show()

### 4.4 Evolution des six statistiques individuelles

On compare les moyennes de PV, Attaque, Defense, Attaque Speciale, Defense Speciale et Vitesse.

In [ ]:
stat_evolution = df.groupby('generation')[stat_cols].mean()

stat_evolution.plot(marker='o', figsize=(13, 6))
plt.title('Evolution des statistiques individuelles par generation')
plt.xlabel('Generation')
plt.ylabel('Statistique moyenne')
plt.xticks(range(1, 10))
plt.legend(title='Statistique')
plt.show()

## 5. Analyse des legendaires et fabuleux

Les Pokemon legendaires et fabuleux possedent souvent des statistiques elevees. On les isole afin de verifier s'ils influencent les tendances observees.

In [ ]:
legendary_summary = df.groupby(['generation', 'legendary_status'])['BST'].mean().unstack()
display(legendary_summary.round(2))

sns.boxplot(data=df, x='legendary_status', y='BST', hue='legendary_status', legend=False)
plt.title('BST selon le statut du Pokemon')
plt.xlabel('Categorie')
plt.ylabel('BST')
plt.show()

# Analyse sans legendaires ni fabuleux
df_normal = df[df['legendary_status'] == 'Normal'].copy()
normal_summary = df_normal.groupby('generation').agg(
    BST_mean=('BST', 'mean'),
    specialization_mean=('specialization', 'mean'),
    count=('id', 'count')
).round(2)

display(normal_summary)

## 6. Tests statistiques

Nous utilisons deux approches complementaires :

- Kruskal-Wallis : determine si au moins une generation presente une distribution differente des autres.
- Spearman : mesure l'association monotone entre le numero de generation et les statistiques.

Le seuil de significativite est fixe a 5 %.

In [ ]:
groups_bst = [
    group['BST'].values
    for _, group in df.groupby('generation')
]

kw_bst = stats.kruskal(*groups_bst)

rho_bst, p_bst = stats.spearmanr(
    df['generation'], df['BST']
)

print('TEST BST')
print('Kruskal-Wallis :', kw_bst)
print('Spearman rho :', round(rho_bst, 4))
print('p-value Spearman :', p_bst)

print('\nInterpretation :')
if p_bst < 0.05:
    print('Association significative entre generation et BST.')
else:
    print('Pas d association monotone significative entre generation et BST.')

if rho_bst > 0:
    print('La tendance observee est positive.')
elif rho_bst < 0:
    print('La tendance observee est negative.')
else:
    print('Aucune tendance monotone observee.')

In [ ]:
groups_spec = [
    group['specialization'].values
    for _, group in df.groupby('generation')
]

kw_spec = stats.kruskal(*groups_spec)

rho_spec, p_spec = stats.spearmanr(
    df['generation'], df['specialization']
)

print('TEST SPECIALISATION')
print('Kruskal-Wallis :', kw_spec)
print('Spearman rho :', round(rho_spec, 4))
print('p-value Spearman :', p_spec)

if p_spec < 0.05:
    print('Association significative entre generation et specialisation.')
else:
    print('Pas d association monotone significative entre generation et specialisation.')

print('\nAnalyse sans legendaires et fabuleux :')
rho_normal, p_normal = stats.spearmanr(
    df_normal['generation'], df_normal['BST']
)
print('Spearman BST :', round(rho_normal, 4), 'p-value :', p_normal)

## 7. Identification des profils les plus specialises

On recherche les Pokemon dont le profil presente le plus grand ecart relatif entre les statistiques.

In [ ]:
display(
    df.nlargest(15, 'specialization')[
        ['name', 'generation', 'BST', 'specialization'] + stat_cols
    ].reset_index(drop=True)
)

## 8. Conclusion automatique

Les resultats suivants sont calcules directement a partir du dataset recupere. Ils devront etre interpretes avec les graphiques et les limites methodologiques.

In [ ]:
first_gen = summary.loc[1]
last_gen = summary.loc[9]

variation_bst = (last_gen['BST_mean'] / first_gen['BST_mean'] - 1) * 100
variation_spec = (last_gen['specialization_mean'] / first_gen['specialization_mean'] - 1) * 100

print('========== CONCLUSION ==========')
print(f'BST moyen generation 1 : {first_gen["BST_mean"]:.2f}')
print(f'BST moyen generation 9 : {last_gen["BST_mean"]:.2f}')
print(f'Evolution relative : {variation_bst:.2f} %')

print('\nSpecialisation moyenne generation 1 :', round(first_gen['specialization_mean'], 4))
print('Specialisation moyenne generation 9 :', round(last_gen['specialization_mean'], 4))
print(f'Evolution relative : {variation_spec:.2f} %')

print('\nTest de tendance BST :')
print('rho =', round(rho_bst, 3), '| p =', round(p_bst, 5))

print('\nTest de tendance specialisation :')
print('rho =', round(rho_spec, 3), '| p =', round(p_spec, 5))

print('\nA INTERPRETER :')
if rho_bst > 0 and p_bst < 0.05:
    print('Les generations sont associees a une hausse significative du BST.')
elif p_bst < 0.05:
    print('Les generations sont associees a une baisse significative du BST.')
else:
    print('Aucune tendance monotone significative du BST n est detectee.')

if rho_spec > 0 and p_spec < 0.05:
    print('La specialisation augmente significativement.')
elif p_spec < 0.05:
    print('La specialisation diminue significativement.')
else:
    print('Aucune tendance monotone significative de specialisation n est detectee.')

print('\nAttention : correlation ne signifie pas causalite.')

## 9. Limites de l'etude

- Les statistiques de base ne representent pas a elles seules la puissance reelle en combat.
- Les types, talents, attaques, objets et mecaniques de jeu ne sont pas pris en compte.
- Les generations ne contiennent pas toutes le meme nombre de Pokemon.
- Les legendaires et fabuleux peuvent influencer les moyennes.
- Le coefficient de variation mesure la dispersion des statistiques, pas directement la specialisation strategique.
- Les resultats sont descriptifs et ne permettent pas d'affirmer qu'une generation cause une augmentation des statistiques.

## Sources
- PokéAPI : https://pokeapi.co/docs/v2
- Pandas : https://pandas.pydata.org/docs/
- SciPy : https://docs.scipy.org/doc/scipy/

**Fin du projet.**